# Amazon Price Tracker

We will do a tracker of a product price to check if the price goes down and with that advise the user that is on sale

In [2]:
# First we are going yo use a clono of amazon's page

# https://appbrewery.github.io/instant_pot/


#We want to get the price of the product so lets do it

from bs4 import BeautifulSoup
import requests

In [4]:
#Get te data from the webpage
response = requests.get(url='https://appbrewery.github.io/instant_pot/')
response.raise_for_status()
data = response.text

In [6]:
#DO webscrapping
soup = BeautifulSoup(data, 'html.parser')

In [27]:
#The completed price
#(a-price-whole, a-price-decimal,) this is in the first class, the other one is only the fraction a-price-fraction
#with find 
price = soup.find(name='span', class_='a-price-whole').get_text()
fraction = soup.find(name='span', class_='a-price-fraction').get_text()
print(float(price+fraction))
#with select
price = soup.select_one(selector='span.a-price-whole').get_text()
fraction = soup.select_one(selector='span.a-price-fraction').get_text()
print(float(price+fraction))

99.99
99.99


In [28]:
#there is another tag that have all at once
price = soup.select_one(selector='span.aok-offscreen').get_text()
#the problem is that put a (blank space-$number.number-blank space)
print(price)
print(type(price))
#the strings are indexables, so we can use slice
#the two first elements are the blank space and the symbol
#the last element is a blank space so we start on the one before that
print(price[2:-1])
#again is a string
print(type(price[2:-1]))
#so we make it a float
print(float(price[2:-1]))

 $99.99 
<class 'str'>
99.99
<class 'str'>
99.99


In [ ]:
#all in once
price = float((soup.select_one(selector='span.aok-offscreen').get_text())[2:-1])
print(price)
#and we can do operations with that, to confirm the data is correct 
print(price+5)

99.99
104.99


# Instead of using an email to send the notifications we will use telegram bot

In [37]:
from pathlib import Path
from dotenv import load_dotenv
import os

In [38]:
env_path = Path.cwd().parent/'.env'
load_dotenv(env_path)
API_KEY = os.getenv('API_Telegram_KEY')
ID = os.getenv("CHAT_ID")

In [39]:

#To do the notification we need to set the price if is less than N amount
def send_message(bot_message):
    url = f"https://api.telegram.org/bot{API_KEY}/sendMessage"
    params = {
        "chat_id": ID,
        "text": bot_message,
    }
    response = requests.post(
        url,
        json=params,
        timeout=15
    )
    print("Telegram status:", response.status_code)
    print("Telegram response:", response.json()['ok'])
    response.raise_for_status()

In [40]:
if price<=100:
    send_message(f'The price of the product you wanted is {price}')
else:
    print('There is no updated for the price of your product')

Telegram status: 200
Telegram response: True


# That works fine, but we can improve our code, now making it live website

In [94]:
#live amazon web
URL = 'https://www.amazon.com.mx/Harbinger-Acolchadas-algod%C3%83%C6%92%C3%82%C2%B3n-Levantamiento-mu%C3%83%C6%92%C3%82%C2%B1eca/dp/B001AMGOGQ/?_encoding=UTF8'

In [ ]:
response = requests.get(URL)
response.raise_for_status()
data= response.text

In [96]:
print(data)

<!doctype html><html lang="es-mx" class="a-no-js" data-19ax5a9jf="dingo"><!-- sp:feature:head-start -->
<head><script>var aPageStart = (new Date()).getTime();</script><meta charset="utf-8">
<!-- sp:end-feature:head-start -->
<!-- sp:feature:csm:head-open-part1 -->

<script type="text/javascript">var ue_t0=ue_t0||+new Date();</script>
<!-- sp:end-feature:csm:head-open-part1 -->
<!-- sp:feature:cs-optimization -->
<meta http-equiv="x-dns-prefetch-control" content="on">
<link rel="preconnect" href="https://images-na.ssl-images-amazon.com" crossorigin>
<link rel="preconnect" href="https://m.media-amazon.com" crossorigin>
<!-- sp:end-feature:cs-optimization -->
<!-- sp:feature:csm:head-open-part2 -->
<script type="text/javascript">
window.ue_ihb = (window.ue_ihb || window.ueinit || 0) + 1;
if (window.ue_ihb === 1) {

var ue_csm = window,
    ue_hob = +new Date();
(function(d){var e=d.ue=d.ue||{},f=Date.now||function(){return+new Date};e.d=function(b){return f()-(b?0:d.ue_t0)};e.stub=functio

### The problem with this code is that does not work the same as the other one, why? well when we execute requests, this does not execute the JavaScript as a normal broswer, with that on mind we will have another data that will not have the information that we wanted, so...

**How we can fix that?**

In [97]:
import requests
from bs4 import BeautifulSoup


## We need to do some steps so first we will fix the url

In [ ]:
URL= 'https://www.amazon.com.mx/dp/B001AMGOGQ'

## And also *the magic* is on the right header, with that we will get the right data

In [ ]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/129.0.0.0 Safari/537.36", # Mimics a real browser
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Accept-Language": "es-MX,es;q=0.9", #can change this depending the zone
    "Referer": "https://www.google.com/", # Useful for initial requests
    "DNT": "1",
    "Connection": "keep-alive"
} 

## SO our first attemp with the modifies

In [99]:
response = requests.get(
    URL,
    headers=headers,
    timeout=20
)

print("Status code:", response.status_code)
print("Final URL:", response.url)
print("Content type:", response.headers.get("Content-Type"))
print("Encoding:", response.encoding)

soup = BeautifulSoup(response.content, "html.parser")

print("Page title:",
      soup.title.get_text(strip=True) if soup.title else "No title")

print("HTML preview:")
print(response.text[:1000])

response.raise_for_status()

Status code: 200
Final URL: https://www.amazon.com.mx/dp/B001AMGOGQ
Content type: text/html;charset=UTF-8
Encoding: UTF-8
Page title: Harbinger Correas Acolchadas de algodÃƒ³n para Levantamiento de Pesas con muÃƒ±eca Acolchada NeoTek (par), Merlot : Amazon.com.mx: Deportes y Aire Libre
HTML preview:
<!doctype html><html lang="es-mx" class="a-no-js" data-19ax5a9jf="dingo"><!-- sp:feature:head-start -->
<head><script>var aPageStart = (new Date()).getTime();</script><meta charset="utf-8"/>
<!-- sp:end-feature:head-start -->
<!-- sp:feature:csm:head-open-part1 -->

<script type='text/javascript'>var ue_t0=ue_t0||+new Date();</script>
<!-- sp:end-feature:csm:head-open-part1 -->
<!-- sp:feature:cs-optimization -->
<meta http-equiv='x-dns-prefetch-control' content='on'>
<link rel="preconnect" href="https://images-na.ssl-images-amazon.com" crossorigin>
<link rel="preconnect" href="https://m.media-amazon.com" crossorigin>
<!-- sp:end-feature:cs-optimization -->
<!-- sp:feature:csm:head-open-par

# The problem has to be with the enconding so we are going to use the one that returns

In [100]:
response = requests.get(URL,  headers=headers, timeout=20)
response.raise_for_status()

# Encoding declared in the HTML
response.encoding = "utf-8"

soup = BeautifulSoup(response.text, "html.parser")

# Extract the product title
title = soup.select_one("#productTitle")

if title:
    print(title.get_text(strip=True))
else:
    print("Product title not found")

#As we can see the result is correct (the title shows without problem)

Harbinger Correas Acolchadas de algodón con muñeca Acolchada NeoTek (par)


## If we check all the span tags, we will see now that one will be our price that we are looking for

In [101]:
prices = soup.select("span.a-price span.a-offscreen")

for price in prices:
    print(price.get_text(strip=True))

$279.99
$939.00
$147.00
$169.00
$349.00
$343.96
$343.96
$343.96
$343.96

$85.99
$343.96
$358.97
$521.55
$00
$349.00
$347.50
null
$450.00
$279.99
$699.99
$430.00
$364.99
$279.99
$515.00
$369.00
$189.99
$349.00
$450.00
$430.00
$193.99
$349.99
$298.00
$99.00
$626.00


## After a little research this is the one

In [102]:
print(soup.select_one("span#apex-pricetopay-accessibility-label"))
print(soup.select_one("span#apex-pricetopay-accessibility-label").get_text())
price = float(soup.select_one("span#apex-pricetopay-accessibility-label").get_text()[2:-1])
#thats it
print(price)

<span class="aok-offscreen" data-pricetopay-label="{priceToPay}" data-pricetopay-savings-label="{priceToPay} con un ahorro del {savings}%" id="apex-pricetopay-accessibility-label"> $343.96 </span>
 $343.96 
343.96


# Lets refactor

In [103]:
from bs4 import BeautifulSoup
import requests
from pathlib import Path
from dotenv import load_dotenv
import os

In [104]:
env_path = Path.cwd().parent/'.env'
load_dotenv(env_path)
API_KEY = os.getenv('API_Telegram_KEY')
ID = os.getenv("CHAT_ID")
URL= 'https://www.amazon.com.mx/dp/B001AMGOGQ'
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/129.0.0.0 Safari/537.36", # Mimics a real browser
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Accept-Language": "es-MX,es;q=0.9", #can change this depending the zone
    "Referer": "https://www.google.com/", # Useful for initial requests
    "DNT": "1",
    "Connection": "keep-alive"
} 

In [105]:
def get_data():
    response = requests.get(URL,  headers=headers, timeout=20)
    response.raise_for_status()

    # Encoding declared in the HTML
    response.encoding = "utf-8"
    data_text = response.text
    return data_text

In [114]:
def scrapping(data):
    soup = BeautifulSoup(response.text, "html.parser")
    # Extract the product title
    title = soup.select_one("#productTitle").get_text()
    #and the price
    price = float(soup.select_one("span#apex-pricetopay-accessibility-label").get_text()[2:-1])
    return title, price    

In [111]:
def send_message(bot_message):
    url = f"https://api.telegram.org/bot{API_KEY}/sendMessage"
    params = {
        "chat_id": ID,
        "text": bot_message,
    }
    response = requests.post(
        url,
        json=params,
        timeout=15
    )
    #print("Telegram response:", response.json()['ok'])
    response.raise_for_status()
    print("Check your inbox telegram")

In [ ]:
def main():
    print("Starting the searching for the price")
    expensive = 345
    data = get_data()
    title, price = scrapping(data)
    #the condition
    print("Comparing...")
    if price<expensive:
        send_message(f'The product:{title}\n Is now on ${price}, (price before ${expensive}), go get it')
    else:
        print('No sales availables')
    print('See you next time')

In [118]:
main()

Starting the searching for the price
Comparing...
Check your inbox telegram
See you next time
